# Week 6a: Fine-Tuning — Theory and Foundations

**Applied Generative AI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

---

## Learning Objectives

By the end of this notebook, you will be able to:

- **Decide when to fine-tune** vs. prompt engineering vs. RAG for your use case
- **Understand full vs. parameter-efficient fine-tuning** — trade-offs in compute, memory, and performance
- **Explain LoRA and QLoRA mathematically** — low-rank decomposition, quantization, and why they work
- **Curate high-quality training datasets** — quality vs. quantity, contamination risks, and best practices
- **Apply a decision framework** to choose the right adaptation strategy for your project

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q transformers datasets torch peft bitsandbytes evaluate accelerate

In [ ]:
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import torch

---
## 1. When to Fine-Tune: Decision Framework

Before investing in fine-tuning, consider whether it's the right approach. Three main strategies:

| Strategy | Best For | Pros | Cons |
|----------|----------|------|------|
| **Prompt Engineering** | General tasks, quick prototyping | No training, instant iteration | Limited by context length, brittle |
| **RAG (Retrieval-Augmented Generation)** | Knowledge-heavy, frequently updated data | No retraining for new docs, traceable | Retrieval quality limits output |
| **Fine-Tuning** | Domain-specific behavior, style, format | Learns new patterns, smaller prompts | Needs data, compute, risk of forgetting |

**Fine-tune when:**
- You have sufficient labeled or instruction data (hundreds to thousands of examples)
- The task requires consistent output format or domain-specific terminology
- Prompt engineering and RAG have hit their limits
- You need lower latency or smaller models for deployment

---
## 2. Full Fine-Tuning

**Full fine-tuning** updates every parameter of the pre-trained model. It is the most expressive but also the most expensive in compute and memory.

### 2.1 Load Dataset: IMDb Sentiment Classification

We use the **IMDb** dataset: 50K movie reviews labeled as positive (1) or negative (0). This is a classic binary sentiment classification task.

In [ ]:
dataset = load_dataset("imdb")

# Inspect structure
print("Dataset splits:", dataset)
print("\nTrain size:", len(dataset["train"]), "| Test size:", len(dataset["test"]))

### 2.2 Explore Label Distribution

A balanced dataset helps the model learn both classes equally. IMDb is roughly balanced.

In [ ]:
train_labels = dataset["train"]["label"]
test_labels = dataset["test"]["label"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(train_labels, bins=[-0.5, 0.5, 1.5], rwidth=0.8)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Negative', 'Positive'])
axes[0].set_title("Train Label Distribution")

axes[1].hist(test_labels, bins=[-0.5, 0.5, 1.5], rwidth=0.8, color='orange')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Negative', 'Positive'])
axes[1].set_title("Test Label Distribution")

plt.tight_layout()
plt.show()

print("\nSample training examples:")
for i in range(3):
    lbl = "positive" if dataset['train']['label'][i] == 1 else "negative"
    print(f"Label: {lbl} | Text: {dataset['train']['text'][i][:150]}...")

### 2.3 Tokenization

We use **DistilBERT** — a lighter BERT variant. Tokenization converts text to subword IDs. Key choices:
- `padding="max_length"`: Pad to fixed length (512 for BERT) for batch processing
- `truncation=True`: Cut longer sequences to avoid overflow

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# Use subsets for faster experimentation (full dataset for production)
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(5000))
small_test = tokenized_datasets["test"].shuffle(seed=42).select(range(2000))

print("Train samples:", len(small_train), "| Eval samples:", len(small_test))

### 2.4 Load Model and Define Metrics

**DistilBertForSequenceClassification** adds a classification head on top of the encoder. The head is randomly initialized and trained from scratch.

**Why 2e-5 learning rate?** Pre-trained weights are already good; a small LR prevents catastrophic forgetting. Typical range: 1e-5 to 5e-5.

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

### 2.5 Training Arguments — What Matters

| Argument | Role | Typical Values |
|----------|------|----------------|
| `num_train_epochs` | How many passes over data | 2–5 for small datasets |
| `per_device_train_batch_size` | Samples per GPU step | 8–32 (limited by GPU memory) |
| `learning_rate` | Step size for updates | 1e-5 to 5e-5 for BERT-like models |
| `weight_decay` | L2 regularization | 0.01–0.1 to reduce overfitting |
| `logging_steps` | How often to log | 50–100 |

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation:", eval_results)

### 2.6 Inference Example

After training, the model can classify new text. The classification head outputs logits; we take argmax for the predicted class.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

test_texts = [
    "I absolutely loved this movie, it was fantastic!",
    "This was the worst film I've ever seen."
]

tokens = tokenizer(test_texts, padding=True, truncation=True, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**tokens)
    preds = outputs.logits.argmax(dim=-1)

for text, pred in zip(test_texts, preds):
    label = "positive" if pred.item() == 1 else "negative"
    print(f"Review: {text}\nPredicted: {label}\n")

---
## 3. Parameter-Efficient Methods — Theory

Full fine-tuning updates millions of parameters. **Parameter-efficient fine-tuning (PEFT)** updates only a small fraction, reducing memory and compute while often matching performance.

### 3.1 LoRA: Low-Rank Adaptation

**Idea:** Instead of updating the full weight matrix $W \in \mathbb{R}^{d \times k}$, we add a low-rank update:

$$W' = W + \Delta W \quad \text{where} \quad \Delta W = B \cdot A$$

with $A \in \mathbb{R}^{r \times k}$, $B \in \mathbb{R}^{d \times r}$, and **rank** $r \ll \min(d, k)$.

**Why it works:** Pre-trained models often lie in low-dimensional subspaces. Small updates in the right directions suffice. Typical $r$: 4–64.

**Parameters:** We train only $A$ and $B$ — $r \cdot (d + k)$ vs. $d \cdot k$ for full fine-tuning.

```
     W (frozen)          A (trainable)    B (trainable)
   [d x k]          [r x k]          [d x r]
       \                |                 /
        \               |                /
         \___  +  (B @ A)  =  W + ΔW
```

### 3.2 LoRA Hyperparameters

| Parameter | Meaning | Typical Values |
|-----------|----------|-----------------|
| `r` (rank) | Dimension of low-rank update | 4, 8, 16, 32 |
| `alpha` | Scaling factor: output = $\alpha/r \cdot (B \cdot A)$ | Often 2×r |
| `target_modules` | Which layers get LoRA | `["q", "v"]` or `["q_lin", "v_lin"]` for attention |
| `lora_dropout` | Dropout on LoRA path | 0.05–0.1 |

### 3.3 QLoRA: Quantized LoRA

**QLoRA** combines:
1. **4-bit quantization** of the base model (NF4 or GPTQ)
2. **LoRA** adapters in full precision

The base weights are frozen and quantized. Only LoRA matrices are stored and updated in fp16/bf16. This dramatically reduces memory (often 50–75% less) while preserving most of the fine-tuning quality.

**Use case:** Fine-tuning 7B+ models on consumer GPUs (e.g., 24GB VRAM).

### 3.4 Other PEFT Methods (Brief)

| Method | Idea | Pros | Cons |
|--------|------|------|------|
| **Adapters** | Small bottleneck layers inserted between transformer blocks | Modular, easy to swap | Extra latency |
| **Prefix Tuning** | Trainable continuous vectors prepended to keys/values | No changes to base model | Limited capacity |
| **Prompt Tuning** | Soft prompts only | Very parameter-efficient | Weaker for complex tasks |

---
## 4. Dataset Curation

### 4.1 Quality vs. Quantity

- **Quality matters more** for instruction tuning and specialized tasks. Noisy labels or inconsistent formats hurt performance.
- **Quantity** helps for broad coverage, but diminishing returns after a certain point.
- **Rule of thumb:** Start with 500–2000 high-quality examples; scale only if needed.

### 4.2 Data Contamination Risks

**Contamination** = test/benchmark data appearing in pre-training or fine-tuning data.

- Inflates reported metrics
- Use deduplication (e.g., n-gram overlap, embedding similarity)
- Prefer held-out validation sets that were never seen during curation

### 4.3 Best Practices

1. **Consistent format** — Same structure for all examples (e.g., instruction/input/output)
2. **Diverse examples** — Cover edge cases, different phrasings
3. **Balanced labels** — Avoid severe class imbalance
4. **Human review** — Spot-check for errors and bias

---
## 5. Exercises

### Exercise 1: Decision Framework

For each scenario, decide: **Prompt Engineering**, **RAG**, or **Fine-Tuning**? Justify briefly.

1. Customer support chatbot that must answer from a 100-page internal FAQ
2. Model that generates poetry in the style of Emily Dickinson
3. General-purpose assistant that occasionally needs up-to-date stock prices

### Exercise 2: Dataset Quality Audit

Load a small subset of IMDb and audit it:
- Check for duplicate or near-duplicate reviews
- Inspect label consistency (do positive reviews actually sound positive?)
- Report any issues you find

In [ ]:
# Dataset quality audit — your code here
audit_subset = dataset["train"].select(range(100))

# Example: Check a few samples for label consistency
for i in [0, 10, 50]:
    text = audit_subset["text"][i][:80]
    label = "positive" if audit_subset["label"][i] == 1 else "negative"
    print(f"[{label}] {text}...")

---
## 6. Responsible AI Reflection

**Bias amplification in fine-tuned models:**

Fine-tuning on biased data can **amplify** existing biases. The model learns to reproduce and sometimes exaggerate patterns in the training set.

- **Mitigation:** Audit training data for demographic skew, stereotypical associations, and toxic content
- **Evaluation:** Use fairness metrics (e.g., disparate performance across groups) and bias benchmarks
- **Documentation:** Record data sources, labeling procedures, and known limitations

---
## 7. Summary & Next Steps

**You learned:**
- When to fine-tune vs. prompt vs. RAG
- Full fine-tuning with DistilBERT on IMDb
- LoRA/QLoRA theory and parameter-efficient methods
- Dataset curation and contamination risks

**Next:** In **Week 6b**, you will apply LoRA hands-on, use WandB for experiment tracking, and evaluate fine-tuned models systematically.